# Import Pyspark

In [0]:
from pyspark.sql.functions import * 

# Create a widget for incremental run

In [0]:
dbutils.widgets.text('incremental flag','0')

# Check incremental run

In [0]:
incremental_flag=dbutils.widgets.get('incremental flag')
incremental_flag

'0'

# Stablish conncetion with ADLS

In [0]:
spark.conf.set('fs.azure.account.key.byanythingstorageaccount.dfs.core.windows.net','Rmv4nXVTTn5udrwp7Lorm3fQgQyhrBzYf7R6OWfggdR5BxgjBEzSoXs7qQ7JVx/4zJY8MAMsNoQG+AStm1PdYw==')

In [0]:

all_files = dbutils.fs.ls('abfss://siilver@byanythingstorageaccount.dfs.core.windows.net')

parquet_files = [f for f in all_files if f.name.endswith('.parquet')]


if len(parquet_files) > 0:
    
    latest_file_obj = sorted(parquet_files, key=lambda x: x.modificationTime, reverse=True)[0]
    filename = latest_file_obj.name
    
    print(f"Loading the latest parquet file: {filename}")
    
    
    
else:
    print("No parquet files found in the silver container!")

Loading the latest parquet file: part-00000-tid-7572864890151711022-9cadb568-c3fd-4975-99a6-589bd35e6546-3000-1-c000.snappy.parquet


##Load the file

In [0]:
df=spark.read.parquet(f'abfss://siilver@byanythingstorageaccount.dfs.core.windows.net/{filename}')
df.display()

OrderID,OrderDate,CustomerID,CustomerName,Country,ProductID,ProductCategory,ProductName,Quantity,UnitPrice,TotalPrice,SalesRegion
12.0,2024-11-12,5174.0,Larry Mack,Croatia,810.0,Clothing,T-shirt,3.0,46.61,139.83,East
171.0,2024-08-05,7143.0,Timothy French,Ireland,535.0,Electronics,Smartphone,3.0,99.14,297.42,West
276.0,2024-10-24,7417.0,Nicole Moses,Sweden,797.0,Sports,Basketball,10.0,585.22,5852.2,West
807.0,2024-07-28,7351.0,Joshua Rojas,Lebanon,535.0,Sports,Football,2.0,680.94,1361.88,North
183.0,2024-03-13,7107.0,Randall Robinson,Senegal,556.0,Furniture,Chair,6.0,530.03,3180.18,North
208.0,2025-03-25,4902.0,Kenneth Meyer,Ireland,922.0,Beauty,Nail Polish,7.0,146.1,1022.7,East
511.0,2024-08-06,7766.0,Anthony Houston,Marshall Islands,982.0,Sports,Basketball,2.0,935.11,1870.22,West
379.0,2023-12-28,9420.0,Brianna Yoder,Latvia,984.0,Furniture,Table,7.0,401.11,2807.77,South
781.0,2025-04-01,9387.0,Benjamin Hardy,Mexico,899.0,Clothing,Shoes,8.0,329.99,2639.92,West
889.0,2023-10-26,5017.0,James Henderson,Monaco,534.0,Beauty,Nail Polish,5.0,211.76,1058.8,West


# Working on Sales Region dimension

In [0]:
%sql 
select distinct(SalesRegion)  from parquet.`abfss://siilver@byanythingstorageaccount.dfs.core.windows.net/`

SalesRegion
null
South
East
West
North


# Create DataFrame - df_src

In [0]:
df_src=spark.sql('''
                 select distinct(SalesRegion)  from parquet.`abfss://siilver@byanythingstorageaccount.dfs.core.windows.net/`
                 ''')
df_src.display()

SalesRegion
null
South
East
West
North


In [0]:
df_src = df_src.na.fill({"SalesRegion": "Central"})
df_src.display()

SalesRegion
Central
South
East
West
North


# Getting Gold layer Data - df_sink

In [0]:
from delta.tables import DeltaTable
path= 'abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_SalesRegion'
if DeltaTable.isDeltaTable(spark,path):

    df_sink=spark.sql('''
        select dim_SalesRegion_Key,SalesRegion from delta.`abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_SalesRegion`
    ''')
else:
    df_sink=spark.createDataFrame([],schema='Dim_SalesRegion_Key string, SalesRegion string')
df_sink.display()

Dim_SalesRegion_Key,SalesRegion


## df_src and df_sink record left join

In [0]:
df=df_src.join(df_sink,df_src['SalesRegion']==df_sink['SalesRegion'],'left').select(df_src['SalesRegion'],df_sink['Dim_SalesRegion_Key'])
df.display()

SalesRegion,Dim_SalesRegion_Key
South,null
Central,null
East,null
West,null
North,null


# filter Old records and New records

In [0]:
df_old=df.filter(df['Dim_SalesRegion_Key'].isNotNull())
df_old.display()

SalesRegion,Dim_SalesRegion_Key


In [0]:
df_new=df.filter(df['Dim_SalesRegion_Key'].isNull())
df_new.display()

SalesRegion,Dim_SalesRegion_Key
Central,null
South,null
East,null
West,null
North,null


# Get maximum Dim_Customer_Key

In [0]:
from pyspark.sql.functions import max

if incremental_flag == 'o':
    max_value = 1
else:
    
    max_value = (df_old.select(max('Dim_SalesRegion_Key')).collect()[0][0] or 0)+1

print(max_value)

1


# Creating Surrogate Key

In [0]:
df=df.withColumn('Dim_SalesRegion_Key',max_value+monotonically_increasing_id())
df.display()

SalesRegion,Dim_SalesRegion_Key
Central,1
South,2
East,3
West,4
North,5


In [0]:
df=df_old.union(df)
df.display()

SalesRegion,Dim_SalesRegion_Key
Central,1
South,2
East,3
West,4
North,5


# SCD Type 1  - Upsart

In [0]:
from delta.tables import DeltaTable

table_name = 'gold.dim_salesregion'
path = 'abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_salesregion'

if DeltaTable.isDeltaTable(spark,path):
    deltaTable = DeltaTable.forPath(spark, path)
    deltaTable.alias('target').merge(
        df.alias('source'),
        'target.Dim_SalesRegion_Key = source.Dim_SalesRegion_Key'
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print('good')
else:
    df.write.format('delta').mode('overwrite').option('mergeSchema', 'true').save(path)
    print('The table is created')

good


In [0]:
%sql
select * from delta.`abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_salesregion`

SalesRegion,Dim_SalesRegion_Key
Central,1
South,2
East,3
West,4
North,5
